In [ ]:
pip install sktime scikit-learn numpy pandas

In [ ]:
import numpy as np
import pandas as pd
from sktime.transformations.panel.rocket import Rocket
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelBinarizer


In [ ]:
file_path = "train_df_ohlc (1).csv"
trainDataset = pd.read_csv(file_path, index_col=[0, 1])

file_path = "test_df_ohlc (2).csv"
testDataset = pd.read_csv(file_path, index_col=[0, 1])


In [ ]:
trainDataset

Open      High       Low     Close    Volume  Pattern
Instance Time                                                           
0        0     0.503143  0.591194  0.150942  0.264150  0.393172        1
         1     0.421383  0.553458  0.201257  0.371069  0.073789        1
         2     0.484277  1.000000  0.484277  0.817610  0.295154        1
         3     0.666666  0.792452  0.534591  0.716980  0.000000        1
         4     0.716980  0.937106  0.471698  0.496854  0.219163        1
...                 ...       ...       ...       ...       ...      ...
3877     4     0.086956  0.205534  0.000000  0.090909  0.513805        0
         5     0.167985  0.328064  0.053361  0.278657  0.060375        0
         6     0.217391  0.371542  0.146246  0.175890  0.160077        0
         7     0.179841  0.367588  0.124506  0.365611  0.000000        0
         8     0.990118  1.000000  0.383399  0.539526  1.000000        0

[61882 rows x 6 columns]

In [ ]:
import numpy as np
import pandas as pd

def adjust_series_length(group, target_length):

    series = group.values
    current_length = len(series)

    if current_length > target_length:
        return series[:target_length]
    else:
        # Padding with zeros if shorter
        padding = np.zeros((target_length - current_length, series.shape[1]))
        return np.vstack([series, padding])


In [ ]:
trainDataset.columns = trainDataset.columns.str.strip()


In [ ]:
print(trainDataset.columns)


Index(['Open', 'High', 'Low', 'Close', 'Volume', 'Pattern'], dtype='object')


In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']
target = 'Pattern'
series_length = 275  # Target series length for ROCKET

def prepare_rocket_data(dataset, features, target, series_length):
    adjusted = dataset.groupby(level=0).apply(
        lambda group: adjust_series_length(group[features], series_length)
    )

    X = np.stack(adjusted.values)


    y = dataset.groupby(level=0)[target].first().values

    return X, y


X_train, y_train = prepare_rocket_data(trainDataset, features, target, series_length)
X_test, y_test = prepare_rocket_data(testDataset, features, target, series_length)

X_train = np.transpose(X_train, (0, 2, 1))
X_test = np.transpose(X_test, (0, 2, 1))

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


X_train shape: (3878, 5, 275)
y_train shape: (3878,)


In [ ]:
y_train_binary = np.where(y_train == 6, 0, 1)
y_test_binary = np.where(y_test == 6, 0, 1)

In [ ]:
from xgboost import XGBClassifier
from sktime.transformations.panel.rocket import Rocket
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report



rocket = Rocket(num_kernels=7000)
xgb_binary = XGBClassifier()

clf_binary = make_pipeline(rocket, xgb_binary)
clf_binary.fit(X_train, y_train_binary)

# Predict on test set
y_pred_binary = clf_binary.predict(X_test)

# Evaluate performance
print("Binary Classification Report:")
print(classification_report(y_test_binary, y_pred_binary))
print(f"Binary Test Accuracy: {accuracy_score(y_test_binary, y_pred_binary):.4f}")


Binary Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.46      0.61        82
           1       0.95      1.00      0.97       892

    accuracy                           0.95       974
   macro avg       0.93      0.73      0.79       974
weighted avg       0.95      0.95      0.94       974

Binary Test Accuracy: 0.9507


NameError: name 'X_train' is not defined